# S&P 500 Macroeconomic Forecasting
## State-Dependent Supervised Screening & Regularized Factor (SSRF) Architecture

This notebook demonstrates the implementation of the SSRF model for forecasting S&P 500 excess returns using macroeconomic indicators.

### References
- McCracken, M. W. & Ng, S. (2016). FRED-MD Database
- Huang, D. et al. (2022). Scaled PCA
- Campbell, J. Y. & Thompson, S. B. (2008). Campbell-Thompson R² OOS

In [ ]:
# Import required packages
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, '../src')

# Suppress warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully')

## 1. Data Generation

Since we may not have a FRED API key, we'll generate sample data for demonstration purposes.

In [ ]:
from fred_data import generate_sample_data, DataProcessor

# Generate sample macroeconomic data
indicators, target = generate_sample_data(
    n_periods=400,
    n_indicators=50,
    start_date='1960-01-01'
)

print(f'Indicators shape: {indicators.shape}')
print(f'Target shape: {target.shape}')
print(f'Date range: {indicators.index[0].strftime("%Y-%m")} to {indicators.index[-1].strftime("%Y-%m")}')

In [ ]:
# Process data
processor = DataProcessor()
indicators = processor.handle_missing_values(indicators, method='ffill')
indicators = processor.winsorize_outliers(indicators)

# Create feature groups
groups = processor.create_category_groups(indicators)
groups_dict = {k: v.columns.tolist() for k, v in groups.items()}

print('Feature Groups:')
for group, features in groups_dict.items():
    print(f'  {group}: {len(features)} features')

## 2. Data Exploration

In [ ]:
# Plot target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution
axes[0].hist(target.values, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(target.mean(), color='red', linestyle='--', label=f'Mean: {target.mean():.4f}')
axes[0].axvline(0, color='gray', linestyle='-', alpha=0.5)
axes[0].set_xlabel('Monthly Return')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of S&P 500 Monthly Returns')
axes[0].legend()

# Time series
axes[1].plot(target.index, target.values, linewidth=0.5, alpha=0.7)
axes[1].axhline(0, color='gray', linestyle='-', alpha=0.5)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Monthly Return')
axes[1].set_title('S&P 500 Monthly Returns Over Time')

plt.tight_layout()
plt.show()

In [ ]:
# Plot indicator categories
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (cat_name, cat_df) in enumerate(groups.items()):
    if idx < 5:
        # Plot average across indicators in category
        cat_avg = cat_df.mean(axis=1)
        axes[idx].plot(cat_avg.index, cat_avg.values, linewidth=0.5)
        axes[idx].set_title(cat_name.replace('_', ' ').title())
        axes[idx].set_xlabel('Date')

axes[5].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of indicators
sample_corr = indicators.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    sample_corr.iloc[:20, :20],  # Show first 20 for readability
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Matrix of First 20 Indicators')
plt.tight_layout()
plt.show()

## 3. SSRF Model Components

In [ ]:
from ssrf_model import (
    SSRFModel, SSRFConfig, 
    GroupwiseScreen, PredictiveScaler,
    SupervisedFactorExtractor, RegimeProxy
)

print('SSRF Model components imported successfully')

In [ ]:
# Demonstrate each stage of the SSRF pipeline

# Split data for demonstration
train_size = 200
X_train = indicators.iloc[:train_size]
y_train = target.iloc[:train_size]

print('=' * 50)
print('SSRF Pipeline Demonstration')
print('=' * 50)

# Stage 1: Group-wise Screening
print('\n📊 Stage 1: Group-wise Supervised Screening')
screen = GroupwiseScreen(t_stat_threshold=0.75)
screened, selected_features = screen.fit_transform(X_train, y_train, groups_dict)
print(f'Selected {len(selected_features)} features total')

# Stage 2: Predictive Scaling
print('\n📐 Stage 2: Predictive Scaling')
X_screened = pd.concat(screened.values(), axis=1)
scaler = PredictiveScaler()
X_scaled, scales = scaler.fit_transform(X_screened)
print(f'Scaled {X_scaled.shape[1]} features')

# Stage 3: Factor Extraction
print('\n🔢 Stage 3: Supervised Factor Extraction')
factor_extractor = SupervisedFactorExtractor(n_factors=5)
factors = factor_extractor.fit_transform(X_scaled)
print(f'Extracted {factors.shape[1]} factors')

# Stage 4: Regime Proxy
print('\n📈 Stage 4: Regime Proxy & Interaction')
regime = RegimeProxy(regime_window=12)
rolling_vol = regime.compute_rolling_volatility(y_train)
regime_p = regime.compute_percentile_rank(rolling_vol, rolling_vol)
X_final = regime.create_interaction_terms(factors, regime_p)
print(f'Created {X_final.shape[1]} features (main + interactions)')

In [ ]:
# Visualize extracted factors
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot factors
for col in factors.columns[:3]:
    axes[0].plot(factors.index, factors[col].values, label=col, alpha=0.7)
axes[0].set_title('Extracted Latent Factors (First 3)')
axes[0].set_xlabel('Date')
axes[0].legend()

# Plot regime proxy
axes[1].plot(regime_p.index, regime_p.values, color='purple', linewidth=1)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].fill_between(regime_p.index, regime_p.values, 0.5, 
                    where=(regime_p.values > 0.5), alpha=0.3, color='red', label='High Vol')
axes[1].fill_between(regime_p.index, regime_p.values, 0.5, 
                    where=(regime_p.values <= 0.5), alpha=0.3, color='green', label='Low Vol')
axes[1].set_title('Regime Proxy: Rolling 12-Month Volatility Percentile')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Percentile')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Full Model Training

In [ ]:
# Configure and train the full SSRF model
config = SSRFConfig(
    t_stat_threshold=0.75,
    n_factors=5,
    regime_window=12,
    elastic_net_alpha=0.001,
    elastic_net_l1_ratio=0.5,
    use_elastic_net_cv=True
)

model = SSRFModel(config)
model.fit(X_train, y_train, groups_dict)

print(f'\nModel trained successfully')
print(f'Non-zero coefficients: {np.sum(model.model.coef_ != 0)}')

In [ ]:
# Generate predictions on test data
X_test = indicators.iloc[train_size:]
y_test = target.iloc[train_size:]

# For regime proxy, we need returns up to test date
y_for_regime = target.iloc[:train_size]

predictions = model.predict(X_test, y_for_regime)

print(f'Test period: {X_test.index[0].strftime("%Y-%m")} to {X_test.index[-1].strftime("%Y-%m")}')
print(f'Number of predictions: {len(predictions)}')

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Time series comparison
axes[0].plot(y_test.index, y_test.values, label='Actual', alpha=0.7, linewidth=1)
axes[0].plot(predictions.index, predictions.values, label='SSRF Predicted', alpha=0.7, linewidth=1)
axes[0].axhline(0, color='gray', linestyle='-', alpha=0.5)
axes[0].set_title('S&P 500 Monthly Returns: Actual vs Predicted')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Return')
axes[0].legend()

# Scatter plot
axes[1].scatter(y_test.values, predictions.values, alpha=0.5)
max_val = max(abs(y_test.values).max(), abs(predictions.values).max())
axes[1].plot([-max_val, max_val], [-max_val, max_val], 'r--', label='Perfect Prediction')
axes[1].axhline(0, color='gray', linestyle='-', alpha=0.3)
axes[1].axvline(0, color='gray', linestyle='-', alpha=0.3)
axes[1].set_xlabel('Actual Return')
axes[1].set_ylabel('Predicted Return')
axes[1].set_title('Prediction Scatter Plot')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Walk-Forward Backtesting

In [ ]:
from backtesting import WalkForwardBacktester, plot_predictions

print('Running walk-forward backtest...')
print('This may take a few minutes...')

# Run backtest with smaller parameters for speed
backtest_config = SSRFConfig(
    t_stat_threshold=0.75,
    n_factors=5,
    regime_window=12,
    elastic_net_alpha=0.001,
    use_elastic_net_cv=True
)

backtester = WalkForwardBacktester(
    model_class=SSRFModel,
    initial_train_window=120,
    use_ct_restriction=True,
    step_size=3  # Use 3-month steps for faster demo
)

result = backtester.run(
    indicators,
    target,
    groups_dict,
    model_config=backtest_config,
    verbose=False
)

print('Backtest completed!')

In [ ]:
# Print backtest metrics
print('\n' + '=' * 50)
print('BACKTEST RESULTS')
print('=' * 50)

print(f"\nCampbell-Thompson R² OOS: {result.metrics['r2_oos']:.4f}")
print(f"Hit Ratio: {result.metrics['hit_ratio']:.2%}")
print(f"\nSharpe Ratio: {result.metrics['sharpe_ratio']:.4f}")
print(f"Calmar Ratio: {result.metrics['calmar_ratio']:.4f}")
print(f"\nMax Drawdown: {result.metrics['max_drawdown']:.2%}")
print(f"Benchmark Max Drawdown: {result.metrics['benchmark_max_drawdown']:.2%}")
print(f"\nCumulative Return: {result.metrics['cumulative_return']:.2%}")
print(f"Benchmark Return: {result.metrics['benchmark_cumulative_return']:.2%}")

In [ ]:
# Plot backtest results
fig = plot_predictions(result)
plt.show()

## 6. Evaluation Metrics

In [ ]:
from evaluation import MetricsCalculator, StatisticalTests, generate_report

# Calculate comprehensive metrics
calc = MetricsCalculator(annualization_factor=12)

metrics = calc.calculate(
    result.predictions,
    result.actual_returns,
    result.benchmark_predictions
)

# Generate report
report = generate_report(metrics, model_name='SSRF')
print(report)

In [ ]:
# Run statistical tests
dm_stat, dm_pval = StatisticalTests.dm_test(
    result.actual_returns.values,
    result.predictions.values,
    result.benchmark_predictions.values
)

cw_stat, cw_pval = StatisticalTests.cw_test(
    result.actual_returns.values,
    result.predictions.values,
    result.benchmark_predictions.values
)

print('Statistical Significance Tests')
print('=' * 50)
print(f'\nDiebold-Mariano Test:')
print(f'  t-statistic: {dm_stat:.4f}')
print(f'  p-value: {dm_pval:.4f}')
print(f'  Significant: {"Yes" if dm_pval < 0.05 else "No"}')

print(f'\nClark-West Test:')
print(f'  t-statistic: {cw_stat:.4f}')
print(f'  p-value: {cw_pval:.4f}')
print(f'  Significant: {"Yes" if cw_pval < 0.05 else "No"}')

In [ ]:
# R² OOS confidence interval
r2_lower, r2_upper = StatisticalTests.out_of_sample_r2_confidence_interval(
    result.metrics['r2_oos'],
    len(result.predictions)
)

print(f'\nCampbell-Thompson R² OOS 95% CI:')
print(f'  [{r2_lower:.4f}, {r2_upper:.4f}]')

## 7. Model Comparison

In [ ]:
# Compare different model configurations
from backtesting import compare_models

models = {
    'SSRF (5 factors)': (SSRFModel, SSRFConfig(n_factors=5)),
    'SSRF (10 factors)': (SSRFModel, SSRFConfig(n_factors=10)),
    'SSRF (low threshold)': (SSRFModel, SSRFConfig(t_stat_threshold=1.0)),
}

print('Comparing model configurations...')
comparison = compare_models(models, indicators, target, groups_dict)
print(comparison)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = ['r2_oos', 'hit_ratio', 'sharpe_ratio']
titles = ['R² OOS', 'Hit Ratio', 'Sharpe Ratio']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    axes[idx].barh(comparison.index, comparison[metric])
    axes[idx].set_xlabel(title)
    axes[idx].set_title(f'{title} Comparison')
    for i, v in enumerate(comparison[metric]):
        axes[idx].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Conclusion

This notebook demonstrated the implementation of the State-Dependent Supervised Screening & Regularized Factor (SSRF) architecture for S&P 500 return forecasting. Key findings:

1. **SSRF Pipeline**: The four-stage pipeline (screening, scaling, factor extraction, regime interaction) provides a defensible approach to handling the low signal-to-noise ratio in stock return prediction.

2. **Campbell-Thompson R² OOS**: This metric provides a statistically rigorous out-of-sample evaluation that compares against the historical mean benchmark.

3. **Regime-Dependency**: The volatility-based regime proxy captures time-varying market conditions that affect the relationship between macroeconomic indicators and stock returns.

4. **Look-Ahead Bias Prevention**: Using point-in-time data from ALFRED ensures that the model is evaluated in a realistically forward-looking manner.